# MESA Finetune

## Pre-requisites

- MESA Runtime credentials

## Overview

Finetune sits alongside datagen as the second 'build' component of MESA (below). Much of finetune is handy middleware for interacting with underlying libraries (e.g. a HuggingFace estimator via AWS SageMaker, or MLX on Apple Silicon [bare metal]), including handling training data download, storing canonical training configurations, error recovery, and model tag/upload/publish.

<p align="center">
  <img src="https://londonaicentre.github.io/MESA-Build/repo.png" width="700" alt="Flowchart of the document generation pipeline">
</p>

As per our running example, we will download and use the training pairs generated in the datagen notebook, and use these to train a small schema standardisation model. Before that, an overview is provided of the wider training (and inference) architecture also provided by MESA, which can be skipped.

## (Optional) Training (and inference) architecture

Runtime is an in-house orchestration tool to support GPU-based tasks such as inference and training. Runtime leverages GitHub's existing workflow orchestration tool (shown below), with jobs run on in-house compute resources (e.g. a Mac Studio) via remote runners. Each job involves the use of a _software_ runner, which packages up relevant logic: in this case the fine-tuning logic provided by the finetune package.

<p align="center">
  <img src="https://londonaicentre.github.io/mesa-runtime/architecture.svg" width="700" alt="Diagram showing MESA training and inference architecture">
</p>

<p align="center">
  <img src="https://londonaicentre.github.io/mesa-runtime/frontend/ui.png" width="700" alt="Screenshot showing MESA runtime UI">
</p>

Finetuning jobs can be initiated by an admin on GitHub directly, via a custom mesa-runtime UI (above) or via a python library. Build pipelines for existing models (e.g. OncoQwen) leverage this library, and it will be used in the example below.

## Steps

1. Import some utility classes to help us work with the library.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

2. Import the MESA runtime library, and create an object of its primary class with our credentials. 

In [ ]:
from mesa_runtime import MesaRuntime
runtime: MesaRuntime = MesaRuntime(
    os.environ["BASE_URL"], os.environ["USERNAME"], os.environ["PASSWORD"]
)

3. List the training data we have available to work with. We should see the batch we uploaded in the datagen step.

In [ ]:
runtime.list_training_data()

4. We can now start our finetuning job on the remote infrastructure (default: Mac Studio). We supply: 

- The name of the training data in our bucket (Step 3)
- The name of the model we're training
- The schema we've generated our training pairs for (with the word 'schema' truncated)
- A description
- A version for our model.
- The name of the configuration for our fine-tuning job, and a flag to indicate that we want to use a local file for training (rather than a stock one).
- A flag to indicate that we want to push our model somewhere publicly accessible for the next stage of our example.

> ❗Finetuning is a slow-running operation. If you wish to, this step can be skipped and you can move to the next notebook.

In [ ]:
run_id: int = runtime.dispatch(
    "20260706-205744_oncotest-batch",
    model_name="oncotest",
    schema="onco",
    description="Test oncology model",
    version="0.1.0",
    config="config.yaml",
    config_from_file=True,
    push_public=True
)

5. We can check the status of the job by passing the run id output from the previous command. Once it reads as complete, we're ready to move to the next notebook.

In [ ]:
runtime.get_run(run_id=run_id)